# 1. Import Libraries

In [ ]:
import os
import sys
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from datetime import datetime

# 2. File Paths

In [ ]:
scripts_path = os.path.abspath(os.path.join('..', 'Scripts'))
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

data_dir = os.path.abspath(os.path.join('..', 'Data', 'TrainTest'))
save_dir = os.path.abspath(os.path.join('..', 'Data', 'SavedModels'))
hyperparameters_dir = os.path.abspath(os.path.join('..', 'Data', 'HyperParameters'))
visualization_dir = os.path.abspath(os.path.join('..', 'Data', 'Visualization'))
results_dir = os.path.abspath(os.path.join('..', 'Data', 'EvaluationResults'))

# 3. Load Models & Data

## 1. Data

### 1. Train Data

In [ ]:
# Load train data
train_data = np.load(os.path.join(data_dir, 'train_data.npy')).astype(np.float32)

num_users, num_items = train_data.shape
print(f"Dimensi Data: {num_users} Users, {num_items} Items")

### 2. Test Data

In [ ]:
# Load test data
test_data = np.load(os.path.join(data_dir, 'test_data.npy')).astype(np.float32)

# Mengambil nilai yang bukan 0, berarti yang sudah diberi rating oleh pengguna
test_indices = np.where(test_data > 0)

#### 1. Denormalize Test Data

In [ ]:
MAX_RATING = 5.0
# Denormalisasi nilai asli (karena data di-scaling dengan dibagi 5.0 di awal)
actual_values = test_data[test_indices] * MAX_RATING

## 2. Models

In [ ]:
from model import Encoder, Decoder, VAE

# 4. Construct Model

## 1. VAE

In [ ]:
# Load hyperparameter terbaik
vae_progress_file = os.path.join(hyperparameters_dir, 'tuning_progress_vae_mul.json')
with open(vae_progress_file, 'r') as f:
    best_vae_params = json.load(f)['best_params']

# Membangun Arsitektur VAE
encoder = Encoder(hidden_dims=best_vae_params['hidden_dims'], latent_dim=best_vae_params['latent_dim'], dropout_rate=best_vae_params['dropout_rate'])
decoder = Decoder(hidden_dims=best_vae_params['hidden_dims'][::-1], output_dim=num_items)
eval_vae = VAE(encoder, decoder, beta=best_vae_params['beta'])

# Menyuntikkan Bobot ke VAE
_ = eval_vae(train_data[:1]) 
eval_vae.load_weights(os.path.join(save_dir, 'trained_best_vae_weights_mul.weights.h5'))

## 2. RSVD

In [ ]:
# Memuat komponen Bias
mu = np.load(os.path.join(save_dir, 'final_mu_mul.npy'))
b_u = np.load(os.path.join(save_dir, 'final_b_u_mul.npy'))
b_i = np.load(os.path.join(save_dir, 'final_b_i_mul.npy'))

# Memuat komponen Laten
U = np.load(os.path.join(save_dir, 'best_U_mul.npy'))
Sigma = np.load(os.path.join(save_dir, 'best_Sigma_mul.npy'))
V = np.load(os.path.join(save_dir, 'best_V_mul.npy'))

# 5. Prediction

## 1. VAE

### 1. Extract Latent Space

In [ ]:
# Ubah train data menjadi tensor
train_data_tf = tf.constant(train_data, dtype=tf.float32)

# Ekstrak matriks laten
Z_mean, Z_log_var = eval_vae.encoder.predict(train_data_tf, verbose=0)

In [ ]:
# Prediksi dari Decoder (skala 0 - 1)
# Decoder Sigmoid - output langsung bisa didenormalisasi ke skala rating
pred_vae_norm = eval_vae.decoder.predict(Z_mean, verbose=0)

### 2. Get Prediction

### 3. Filter Prediction
    FIlter hanya test data saja

In [ ]:
pred_vae_test = pred_vae_norm[test_indices]

### 4. Denormalize
    Kembalikan ke skala asli

In [ ]:
pred_vae_test_denorm = pred_vae_test * MAX_RATING

### 5. Clip Prediction 
    Batasi prediksi agar tidak keluar dari batas rating

In [ ]:
pred_vae_clipped = np.clip(pred_vae_test_denorm, 1.0, 5.0)

## 2. RSVD

### 1. Latent Dot Product

In [ ]:
# Kalkulasi matriks laten
latent_matrix = np.dot(np.dot(U, Sigma), V.T)

### 2. Bias Matrix

In [ ]:
# Menghitung matriks bias (rata-rata global + bias pengguna + bias film)
bias_matrix = float(mu) + b_u[:, np.newaxis] + b_i[np.newaxis, :]

### 3. Combine

In [ ]:
# Gabungkan bias dan interaksi laten menjadi tebakan penuh (Skala 0 - 1)
full_rsvd_pred = bias_matrix + latent_matrix

### 4. Filter Prediction

In [ ]:
# Filter untuk indeks test data saja
pred_rsvd_test = full_rsvd_pred[test_indices]

### 5. Denormalize

In [ ]:
# Kembalikan prediksi ke skala asli
pred_rsvd_test_denorm = pred_rsvd_test * MAX_RATING

### 6. Clip Prediction

In [ ]:
# Batasi prediksi 
pred_rsvd_clipped = np.clip(pred_rsvd_test_denorm, 1.0, 5.0)

## 3. Ensemble

### 1. Weights
    Cari bobot terbaik antara VAE dan RSVD

In [ ]:
best_rmse = float('inf')
best_alpha = 0.0
best_beta = 0.0

# Menyimpan data untuk grafik
alpha_history = []
rmse_history = []

# Menguji alpha dari 0.00 hingga 1.00 dengan rentang 0.01 (1%, 2%, 3%, dst)
alphas = np.linspace(0, 1, 101)

for alpha in alphas:
    beta = 1.0 - alpha  # Total bobot harus selalu 1.0 (100%)
    
    # Gabungkan (Gunakan variabel clipped seperti sebelumnya)
    pred_test = (alpha * pred_vae_clipped) + (beta * pred_rsvd_clipped)
    pred_test_clipped = np.clip(pred_test, 1.0, 5.0)
    
    # Hitung RMSE
    mse_test = np.mean(np.square(actual_values - pred_test_clipped))
    rmse_test = np.sqrt(mse_test)
    
    # Simpan history untuk plot
    alpha_history.append(alpha)
    rmse_history.append(rmse_test)
    
    # Cek apakah ini RMSE terbaik
    if rmse_test < best_rmse:
        best_rmse = rmse_test
        best_alpha = alpha
        best_beta = beta

print(f"Proporsi VAE (Alpha) : {best_alpha * 100:.0f}%  ({best_alpha:.2f})")
print(f"Proporsi RSVD (Beta) : {best_beta * 100:.0f}%  ({best_beta:.2f})")
print(f"RMSE Paling Optimal  : {best_rmse:.4f}")

# Membuat Visualisasi Grafik
plt.figure(figsize=(10, 5))
plt.plot(alpha_history, rmse_history, color='blue', linewidth=2)
plt.axvline(x=best_alpha, color='red', linestyle='--', label=f'Best Alpha: {best_alpha:.2f}')
plt.scatter([best_alpha], [best_rmse], color='red', s=100, zorder=5)

plt.title('Kurva Tuning Bobot Ensemble (VAE vs RSVD)', fontsize=14, pad=15)
plt.xlabel('Proporsi Bobot VAE (Alpha)', fontsize=12)
plt.ylabel('Nilai RMSE (Lebih rendah lebih baik)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()

# Simpan grafiknya agar bisa kamu masukkan ke laporan proposal/skripsi
timestamp = datetime.now().strftime('%Y%m%d_%H%M')
plot_path = os.path.join(visualization_dir, f'ensemble_weight_tuning_{timestamp}.png')
plt.savefig(plot_path, dpi=300)
plt.show()

### 2. Combine

In [ ]:
# Gabung prediksi dengan pembobotan
pred_hybrid = (best_alpha * pred_vae_clipped) + (best_beta * pred_rsvd_clipped)

### 3. Clip Prediction

In [ ]:
# Batasi prediksi
pred_hybrid_clipped = np.clip(pred_hybrid, 1.0, 5.0)

# 6. Evaluation

## 1. Helper Function

In [ ]:
# Fungsi helper untuk menghitung MSE, RMSE, MAE sekaligus
def calculate_metrics(y_true, y_pred):
    mse = np.mean(np.square(y_true - y_pred))
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))
    return mse, rmse, mae

## 2. Calculate Metrics

In [ ]:
mse_v, rmse_v, mae_v = calculate_metrics(actual_values, pred_vae_clipped)       # VAE
mse_r, rmse_r, mae_r = calculate_metrics(actual_values, pred_rsvd_clipped)      # RSVD
mse_h, rmse_h, mae_h = calculate_metrics(actual_values, pred_hybrid_clipped)    # Ensemble

## 3. Comparison

In [ ]:
print(f"Metrik  | VAE Murni   | RSVD Murni  | Ensemble Hybrid Final ")
print("--------|-------------|-------------|-----------------------")
print(f"MSE     | {mse_v:.4f}      | {mse_r:.4f}      | {mse_h:.4f}")
print(f"RMSE    | {rmse_v:.4f}      | {rmse_r:.4f}      | {rmse_h:.4f}")
print(f"MAE     | {mae_v:.4f}      | {mae_r:.4f}      | {mae_h:.4f}")

## 4. Save Results

In [ ]:
# Menyimpan seluruh ringkasan metrik ini agar bisa dipanggil kembali tanpa perlu me-run dari awal
evaluation_results = {
    "timestamp": timestamp,
    "ensemble_weights": {
        "best_alpha_vae": float(best_alpha),
        "best_beta_rsvd": float(best_beta)
    },
    "metrics": {
        "vae_standalone": {"MSE": float(mse_v), "RMSE": float(rmse_v), "MAE": float(mae_v)},
        "rsvd_standalone": {"MSE": float(mse_r), "RMSE": float(rmse_r), "MAE": float(mae_r)},
        "hybrid_ensemble": {"MSE": float(mse_h), "RMSE": float(rmse_h), "MAE": float(mae_h)}
    }
}

evaluation_file_path = os.path.join(results_dir, f'final_testing_results_mul_{timestamp}.json')

with open(evaluation_file_path, 'w') as f:
    json.dump(evaluation_results, f, indent=4)

print(f"\n[SUCCESS] Seluruh hasil evaluasi (Testing Metrics) telah diarsipkan dengan aman ke:")
print(f"-> {evaluation_file_path}")